# Map song genres
2_lb_data_enrichment.ipynb extracted genre data from Musicbrainz based on release_mbids and recording_mbids from the ListenBrainz dataset. However, there are inconsistencies in genre naming, additionally some recordings are missing some tagging.

The first aim of this notebook is to map missing genre data for songs that come from releases that do have partial genre mapping. 

In a second step, tags will be grouped under a more broad genre to reduce the number of genres.

## Load csv into pandas

In the following section, the csv is uploaded and preprocessed dataset in pandas. 

In [14]:
import pandas as pd
import ast
from collections import Counter

EMPTY_TEXT_VALUES = {"", "none", "null", "nan", "na", "n/a", "[]", "{}"}

def clean_text_series(series):
    return series.fillna('').astype(str).str.strip()

def non_empty_text_mask(series):
    return clean_text_series(series).ne('')

full_csv_df = pd.read_csv('output/recordings_subset.csv')

# only keep rows where ISRC is present
csv_isrc_subset = full_csv_df[non_empty_text_mask(full_csv_df['ISRC'])].copy()

csv_isrc_subset.count()


recording_mbid    38765
genres            28272
ISRC              38765
duration          38723
dtype: int64

In [ ]:
df01 = pd.read_parquet('original_files/01_clean.parquet')
df02 = pd.read_parquet('original_files/02_clean.parquet')
df03 = pd.read_parquet('original_files/03_clean.parquet')
df = pd.concat([df01, df02, df03], ignore_index=True)

valid_recording_mbids = set(csv_isrc_subset['recording_mbid'].dropna().astype(str))

df_release_recordings_subset = (
    df[["release_mbid", "recording_mbid"]]
    .dropna(subset=["release_mbid", "recording_mbid"])
    .assign(recording_mbid=lambda d: d['recording_mbid'].astype(str))
    .loc[lambda d: d['recording_mbid'].isin(valid_recording_mbids)]
    .drop_duplicates()
    .groupby("release_mbid", as_index=False)
    .agg({"recording_mbid": list})
)

df_release_recordings_subset.count()


release_mbid      3751
recording_mbid    3751
dtype: int64

## Map missing genre data

Some songs are missing genre, but they may in some cases be on the same releases as songs that do have a genre, the next part of the code maps the top 4 genres of the release for the songs missing genre.

In [ ]:
def parse_genre_set(value):
    if pd.isna(value):
        return set()

    text = str(value).strip()
    if text.lower() in EMPTY_TEXT_VALUES:
        return set()

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        parsed = [part.strip() for part in text.split(',') if part.strip()]

    if isinstance(parsed, str):
        parsed = [parsed]

    if not isinstance(parsed, (list, tuple, set)):
        return set()

    return {str(item).strip().lower() for item in parsed if str(item).strip()}

def parse_genre_list(value):
    return sorted(parse_genre_set(value))

def format_genre_set(genre_set):
    if not genre_set:
        return ''
    return str(sorted(genre_set))

csv_genres_df = full_csv_df.copy()
csv_genres_df['recording_mbid'] = clean_text_series(csv_genres_df['recording_mbid'])
csv_genres_df['genre_set'] = csv_genres_df['genres'].apply(parse_genre_set)

recording_genres = (
    csv_genres_df.groupby('recording_mbid', as_index=False)['genre_set']
    .agg(lambda values: set().union(*values) if len(values) else set())
)

release_recording_pairs = (
    df_release_recordings_subset
    .explode('recording_mbid')
    [['release_mbid', 'recording_mbid']]
    .dropna(subset=['release_mbid', 'recording_mbid'])
    .assign(recording_mbid=lambda d: clean_text_series(d['recording_mbid']))
    .drop_duplicates()
)

release_with_genres = release_recording_pairs.merge(recording_genres, on='recording_mbid', how='left')
release_with_genres['genre_set'] = release_with_genres['genre_set'].apply(
    lambda value: value if isinstance(value, set) else set()
)

release_top_genre_counts = {}
for release_mbid, group in release_with_genres.groupby('release_mbid'):
    counter = Counter()
    for genre_set in group['genre_set']:
        counter.update(genre_set)
    release_top_genre_counts[release_mbid] = Counter(dict(counter.most_common(4)))

recording_releases = (
    release_recording_pairs.groupby('recording_mbid', as_index=False)['release_mbid']
    .agg(list)
)
recording_releases_lookup = dict(zip(recording_releases['recording_mbid'], recording_releases['release_mbid']))

def infer_top_genres_from_album(recording_mbid):
    releases = recording_releases_lookup.get(recording_mbid, [])
    combined = Counter()
    for release_mbid in releases:
        combined.update(release_top_genre_counts.get(release_mbid, Counter()))
    return {genre for genre, _ in combined.most_common(4)}

missing_mask = csv_genres_df['genre_set'].apply(len).eq(0)
inferred_sets = csv_genres_df.loc[missing_mask, 'recording_mbid'].apply(infer_top_genres_from_album)
fillable_mask = inferred_sets.apply(len).gt(0)
csv_genres_df.loc[inferred_sets.index[fillable_mask], 'genre_set'] = inferred_sets[fillable_mask]

# normalize all genre cells: no outer CSV quotes, and keep truly empty rows empty.
csv_genres_df['genres'] = csv_genres_df['genre_set'].apply(format_genre_set)

full_csv_df = csv_genres_df[['recording_mbid', 'genres', 'ISRC', 'duration']].copy()
full_csv_df = full_csv_df[non_empty_text_mask(full_csv_df['ISRC'])].copy()

filled_rows = int(fillable_mask.sum())
print(f"Filled genres for {filled_rows} recordings using album top-4 genre propagation.")

csv_genres_df[['recording_mbid', 'genres', 'ISRC', 'duration']].head()


Filled genres for 2891 recordings using album top-4 genre propagation.


,recording_mbid,genres,ISRC,duration
0,0003dd36-b4d2-4216-a37e-b110f6882ecb,"['alternative rock', 'ambient', 'electronic', ...",GBDCA9900070,491000.0
1,00047577-1669-4da1-9aa5-7d7b7923ebdf,"['alternative rock', 'emo', 'indie', 'indie ro...",US3R49900018,163053.0
2,0006fc51-adb4-4417-b8fb-7b954b853923,"['2-step', 'ambient', 'dubstep', 'electronic',...",GBLZC0500002,301506.0
3,0006ff85-f08f-4c5a-844d-ce0ec70ad670,"['billboard hot 100', 'billboard hot 100 2025'...",USUG12408498,233000.0
4,000dbffe-59b2-42ba-9458-c8989dccaeb9,"['electronic', 'electronica', 'electropop', 'e...",GBAJH0900116,211266.0


## List top genres from the whole dataset 

In [ ]:


full_csv_df['genre_list'] = full_csv_df['genres'].apply(parse_genre_list)
all_genres = [g for sublist in full_csv_df['genre_list'] for g in sublist]
genre_counts = Counter(all_genres)
top_genres = pd.DataFrame(genre_counts.most_common(50), columns=['genre', 'count'])
print(top_genres)


                     genre  count
0                     rock  17724
1               electronic   7449
2                      pop   6484
3         alternative rock   6383
4                    metal   5167
5                 pop/rock   4463
6               indie rock   4111
7              heavy metal   3527
8                    indie   3425
9                     punk   2830
10                pop rock   2700
11  alternative/indie rock   2281
12                 ambient   2244
13               hard rock   2021
14             death metal   1887
15            experimental   1861
16    alternative pop/rock   1826
17               synth-pop   1731
18                 hip hop   1678
19               folk rock   1670
20              soundtrack   1656
21               downtempo   1641
22            classic rock   1469
23               indie pop   1463
24             alternative   1400
25                    folk   1323
26       alternative metal   1290
27                    jazz   1272
28        prog

## Group songs by supergenre

In [ ]:
def map_supergenres(genre_list):
    if not genre_list:
        return ['Other']
    genres_str = ' '.join(genre_list).lower()
    matches = []
    
    if any(word in genres_str for word in ['rock', 'punk', 'alternative', 'indie', 'folk', 'classic', 'progressive rock', 'art', 'psychedelic', 'album']):
        matches.append('Rock')
    if any(word in genres_str for word in ['electronic', 'ambient', 'downtempo', 'synth', 'electro', 'house', 'idm', 'trip hop', 'dance', 'electropop', 'industrial']):
        matches.append('Electronic')
    if 'metal' in genres_str:
        matches.append('Metal')
    if 'pop' in genres_str:
        matches.append('Pop')
    if any(word in genres_str for word in ['hip hop', 'rap']):
        matches.append('Hip-Hop')
    if any(word in genres_str for word in ['jazz', 'folk', 'singer-songwriter']):
        matches.append('Jazz/Folk')
    if any(word in genres_str for word in ['experimental', 'soundtrack', 'score', 'video game', 'vgm', 'modern classical']):
        matches.append('Experimental')
    
    return matches if matches else ['Other']

full_csv_df['supergenres'] = full_csv_df['genre_list'].apply(map_supergenres)
print("Sample:", full_csv_df[['genres', 'supergenres']].head())
full_csv_df[['recording_mbid', 'genres', 'ISRC', 'duration', 'supergenres']].to_csv('output/recordings_subset_with_supergenres.csv', index=False)

# song counts per supergenre
exploded = full_csv_df.explode('supergenres')
print(exploded['supergenres'].value_counts())


Sample:                                               genres  \
0  ['alternative rock', 'ambient', 'electronic', ...   
1  ['alternative rock', 'emo', 'indie', 'indie ro...   
2  ['2-step', 'ambient', 'dubstep', 'electronic',...   
3  ['billboard hot 100', 'billboard hot 100 2025'...   
4  ['electronic', 'electronica', 'electropop', 'e...   

                        supergenres  
0  [Rock, Electronic, Experimental]  
1                            [Rock]  
2                      [Electronic]  
3                         [Hip-Hop]  
4           [Rock, Electronic, Pop]  
Rock            23511
Pop             13845
Electronic       9844
Other            8588
Metal            7148
Jazz/Folk        4997
Experimental     4255
Hip-Hop          2255
Name: supergenres, dtype: int64


# Create the final dataset

In [ ]:
df01 = pd.read_parquet('original_files/01_clean.parquet')
df02 = pd.read_parquet('original_files/02_clean.parquet')
df03 = pd.read_parquet('original_files/03_clean.parquet')
df = pd.concat([df01, df02, df03], ignore_index=True)
print(df.count())

# Only keep rows where recording_mbid is in the csv subset, then add csv metadata columns.
df['recording_mbid'] = clean_text_series(df['recording_mbid'])
csv_lookup = (
    full_csv_df[['recording_mbid', 'genres', 'supergenres', 'duration', 'ISRC']]
    .copy()
    .assign(recording_mbid=lambda d: clean_text_series(d['recording_mbid']))
    .drop_duplicates(subset=['recording_mbid'])
)

valid_recording_mbids = set(csv_lookup['recording_mbid'])
final_dataset = df[df['recording_mbid'].isin(valid_recording_mbids)].copy()


lookup = csv_lookup.set_index('recording_mbid')
for col in ['genres', 'supergenres', 'duration', 'ISRC']:
    final_dataset[col] = final_dataset['recording_mbid'].map(lookup[col])

print(final_dataset.columns)
print(final_dataset.count())


listened_at            2186027
created                2186027
user_id                2186027
recording_msid         2186027
artist_name            2186027
artist_credit_id       2186027
release_name           2186027
release_mbid           2186027
recording_name         2186027
recording_mbid         2186027
artist_credit_mbids    2186027
dtype: int64
Index(['listened_at', 'created', 'user_id', 'recording_msid', 'artist_name',
       'artist_credit_id', 'release_name', 'release_mbid', 'recording_name',
       'recording_mbid', 'artist_credit_mbids', 'genres', 'supergenres',
       'duration', 'ISRC'],
      dtype='object')
listened_at            967970
created                967970
user_id                967970
recording_msid         967970
artist_name            967970
artist_credit_id       967970
release_name           967970
release_mbid           967970
recording_name         967970
recording_mbid         967970
artist_credit_mbids    967970
genres                 967970
supergenr

In [ ]:
from pathlib import Path

output_file = 'output/final_dataset.parquet'

# Compress into a single file
compression_attempts = [('zstd', 12), ('gzip', 9)]
size_report = None

for compression, level in compression_attempts:
    final_dataset.to_parquet(
        output_file,
        index=False,
        engine='pyarrow'
    )

    size_mb = round(Path(output_file).stat().st_size / (1024 ** 2), 2)
    size_report = {
        'file': output_file,
        'rows': len(final_dataset),
        'compression': compression,
        'compression_level': level,
        'size_mb': size_mb,
    }


size_report


{'file': 'output/final_dataset.parquet',
 'rows': 967970,
 'compression': 'gzip',
 'compression_level': 9,
 'size_mb': 59.73}